# Task 1 — MLP Baseline
### Pokemon Type Classification | DL Assignment 1

**Goal:** classify 3600 Pokémon images into 9 types using a flat-pixel MLP. This is the *baseline* — intentionally simple, so the gap with CNN (Task 2) is clear and measurable.  
**Metric:** Macro-averaged F1 — used because class imbalance is 2.76× (Water 674 vs Ground 244). Accuracy would be misleading.  
**Architecture:** `Flatten → FC(12288→512) → BN → ReLU → Dropout → … → FC(128→9)` — ~6.4M params.  
**Imbalance handling:** `CrossEntropyLoss(weight=...)` with inverse-frequency class weights. WeightedRandomSampler optional per experiment.

---
*All reusable logic lives in `src/`. This notebook is the runner + readable story.*  
*All run-level constants are set at the top of Cell 3 — nothing else needs changing.*


## How to run

**Locally** — flip `FAST_RUN = True` (Cell 3), then run all cells. Completes in < 2 min on CPU.  
**Colab** — Cell 3 clones the repo + installs deps. Cell 4 downloads the dataset. Everything else is identical.  
**Switching between runs** — only change the CONFIG block in Cell 3. Nothing else.

**Output files** (all in `task1/outputs/`):
- `plots/` — all EDA plots + training curves + confusion matrix + results comparison chart
- `checkpoints/` — one `.pth` per experiment (best val_loss per run)
- `results/task1_results.json` — full metrics for all experiments (use for report)
- `results/submission_task1.csv` — Kaggle submission (best experiment)


In [ ]:
# ── Cell 3: Setup & CONFIG ────────────────────────────────────────────────────
# All run-level constants live HERE. Change these to switch between a quick
# local smoke-test and a real Colab run — nothing else needs touching.
# ─────────────────────────────────────────────────────────────────────────────

# ┌─── FLIP THIS ONE FLAG ──────────────────────────────────────────────────┐
FAST_RUN = True   # True = smoke-test (tiny data, 2 epochs) | False = real Colab run
# └─────────────────────────────────────────────────────────────────────────┘

# training hyperparams
EPOCHS      = 2     if FAST_RUN else 30
PATIENCE    = 1     if FAST_RUN else 5
LR          = 1e-3
BATCH_SIZE  = 32    if FAST_RUN else 64
IMG_SIZE    = 64    # MLP input: 64x64x3 = 12288 flat features
NUM_WORKERS = 0     if FAST_RUN else 2   # 0 avoids multiprocessing issues locally

# smoke-test data size: 9 classes × N = total training images
N_SAMPLES_PER_CLASS = 6   # 54 total — enough to run the full pipeline in seconds

# ─────────────────────────────────────────────────────────────────────────────
import sys, os, time, json
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not os.path.exists("/content/DL_Proj"):
        !git clone https://github.com/fmssilva/DL_Proj.git /content/DL_Proj
    else:
        !git -C /content/DL_Proj pull --ff-only
    os.chdir("/content/DL_Proj/assignment_1")
    %pip install -r requirements.txt -q
else:
    # notebook kernel starts in task1/ — step up to assignment_1/ root
    cwd = Path(os.getcwd())
    if cwd.name.startswith("task"):
        os.chdir(cwd.parent)

ROOT = os.getcwd()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import torch
import pandas as pd
import matplotlib.pyplot as plt

from src.config import (
    SEED, CLASSES, NUM_CLASSES, DATA_DIR, OUT_DIR,
    get_task_out_dir, set_seed,
)

set_seed(SEED)

TASK_OUT_DIR = get_task_out_dir("task1")
device       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CSV_PATH     = DATA_DIR / "train_labels.csv"
TRAIN_DIR    = DATA_DIR / "Train"
TEST_DIR     = DATA_DIR / "Test"

print(f"ROOT        : {ROOT}")
print(f"FAST_RUN    : {FAST_RUN}  (N_SAMPLES_PER_CLASS={N_SAMPLES_PER_CLASS if FAST_RUN else 'all'})")
print(f"EPOCHS      : {EPOCHS}  |  PATIENCE : {PATIENCE}  |  LR : {LR}")
print(f"BATCH_SIZE  : {BATCH_SIZE}  |  IMG_SIZE : {IMG_SIZE}x{IMG_SIZE}")
print(f"Device      : {device}  |  PyTorch {torch.__version__}")
print(f"Outputs     : {TASK_OUT_DIR.resolve()}")


In [ ]:
# ── Cell 4: Load data ─────────────────────────────────────────────────────────
# Colab: download zip from Google Drive if not already present.
import zipfile

if not Path("data/train_labels.csv").exists():
    if IN_COLAB:
        %pip install gdown -q
        import gdown
        gdown.download(id="1nVSQZxQubLEPXjSRqGn7rtPzkw-S0zIi", output="data.zip", quiet=False)
        with zipfile.ZipFile("data.zip") as zf:
            zf.extractall("data")
        os.remove("data.zip")
        print("data/ ready")
    else:
        print("ERROR: data/ not found — place it under assignment_1/data/")
else:
    print("data/ already present")

df_full = pd.read_csv(CSV_PATH)
print(f"Full dataset : {len(df_full)} rows, {df_full['label'].nunique()} classes")

# FAST_RUN: subsample N per class so every cell finishes fast on CPU
if FAST_RUN:
    df = df_full.groupby("label", group_keys=False).head(N_SAMPLES_PER_CLASS).reset_index(drop=True)
    print(f"FAST_RUN     : subsampled to {len(df)} rows ({N_SAMPLES_PER_CLASS}/class)")
else:
    df = df_full

print(f"\nClass counts used this run:\n{df['label'].value_counts().to_string()}")


---
## Part 1 — Exploratory Data Analysis

EDA always runs on the **full 3600-image training set** (`df_full`), not on the FAST_RUN subset.  
Reason: we want to understand the real data distribution — using a 54-image subset would distort every plot.  
The train/val split happens later (inside `build_loaders`), strictly after EDA.

**What we check:**
1. Class imbalance — drives loss function choice
2. Visual similarity — predicts which classes will be confused
3. Intra-class variance — predicts per-class difficulty
4. Pixel statistics — confirms whether ImageNet normalisation is a good fit
5. Intensity histogram — channel balance + augmentation motivation
6. PCA / t-SNE — is there any linear separation in flat pixel space?
7. Dataset mean/std from train split only — the correct normalisation constants, no leakage


In [ ]:
# ── EDA Stats — run on the full dataset ───────────────────────────────────────
import src.datasets.eda as eda

print("=== Class Distribution ===")
counts_df = eda.class_distribution(df_full)

print("\n=== Image Size Distribution ===")
size_map = eda.image_size_distribution(TRAIN_DIR)

print("\n=== Data Integrity Check ===")
valid, invalid = eda.check_data_integrity(TRAIN_DIR, df_full)
print(f"Result: {valid} valid, {invalid} invalid")


### Plot 1 — Class Distribution

Horizontal bar chart showing the number of samples per class, sorted by frequency.

**What to look for:** the imbalance ratio (max class count ÷ min class count). A ratio above 2× may bias the model toward majority classes — motivating `CrossEntropyLoss(weight=class_weights)`.


In [ ]:
import src.datasets.eda_plots as eda_plots

fig = eda_plots.plot_class_distribution(df_full, out_path=TASK_OUT_DIR / "plots" / "plot_class_distribution.png")
plt.show()
plt.close(fig)


> **Finding — Class Imbalance:**  
> _Fill in after running. Record: majority class name + count, minority class name + count, imbalance ratio (max÷min). Example: "Water (N=168) is the majority class, Ground (N=61) the minority — ratio 2.76×. This motivates inverse-frequency class weighting in CrossEntropyLoss."_


### Plot 2 — Sample Images per Class

4 random images per class (fixed seed — same grid on every run).

**What to look for:** visually similar class pairs that the MLP is likely to confuse. Note colour and texture similarities across classes (e.g. Bug/Grass both greenish; Fighting/Normal both humanoid). These pairs will show up as off-diagonal errors in the confusion matrix.


In [ ]:
fig = eda_plots.plot_sample_images(TRAIN_DIR, df_full, n_per_class=4, out_path=TASK_OUT_DIR / "plots" / "plot_sample_images.png")
plt.show()
plt.close(fig)


> **Finding — Visual Similarity:**  
> _Fill in after running. List 2–3 class pairs that look most similar and explain why the MLP (which ignores spatial layout) will struggle with them. Example: "Bug and Grass share green colouring; the MLP cannot separate them by texture because it treats the image as a flat vector."_


### Plot 3 — Average Image per Class

Mean pixel value across all images in each class (after resizing to 64×64).

**What to look for:** how "blurry" each average image is. A crisp average image means the class has consistent appearance (low intra-class variance → easier to classify). A blurry/washed-out average means high intra-class variance → harder. Note which classes have the sharpest vs blurriest prototypes.


In [ ]:
fig = eda_plots.plot_average_image_per_class(TRAIN_DIR, df_full, out_path=TASK_OUT_DIR / "plots" / "plot_average_image_per_class.png")
plt.show()
plt.close(fig)


> **Finding — Intra-class Variance:**  
> _Fill in after running. Identify the class with the sharpest prototype (low variance, easy to classify) and the one with the blurriest (high variance, hard). Example: "Water has a distinctive blue tone → sharp average. Normal is the most diverse → blurry average → expect lowest per-class F1."_


### Plot 4 — Per-Channel Pixel Statistics

Mean and standard deviation of R, G, B channels across the training set (computed on raw 0–255 values, printed as 0–1 fractions).

**What to look for:** compare your dataset's mean/std to ImageNet's `mean=[0.485, 0.456, 0.406]` / `std=[0.229, 0.224, 0.225]`. Large differences confirm that **dataset-specific normalisation** (used in `get_base_transforms`) is better than reusing ImageNet constants for this task.


In [ ]:
fig = eda_plots.plot_pixel_statistics(TRAIN_DIR, df_full, out_path=TASK_OUT_DIR / "plots" / "plot_pixel_statistics.png")
plt.show()
plt.close(fig)


> **Finding — Normalisation Constants:**  
> _Fill in after running. Record the 6 numeric values printed by the cell (R_mean, G_mean, B_mean, R_std, G_std, B_std) and compare to ImageNet. Example: "Dataset mean=[0.62, 0.58, 0.55], std=[0.18, 0.19, 0.20] — notably brighter and less variable than ImageNet, confirming we should use dataset-specific normalisation."_


### Plot 5 — Pixel Intensity Histogram

Histogram of pixel intensity values (0–255) for R, G, B channels sampled from the training set.

**What to look for:** overall brightness range and channel imbalance. A histogram skewed toward high values → bright/pastel dataset. Nearly overlapping R/G/B histograms → low colour diversity. Either pattern suggests augmentation (colour jitter, flip) would help CNNs in Task 3.


In [ ]:
fig = eda_plots.plot_pixel_intensity_histogram(TRAIN_DIR, df_full, n_samples=200, out_path=TASK_OUT_DIR / "plots" / "plot_pixel_intensity_histogram.png")
plt.show()
plt.close(fig)


> **Finding — Intensity Distribution:**  
> _Fill in after running. Note if the histogram is skewed bright/dark, whether channels overlap heavily, and any channel dominating. Example: "All three channels peak around 180–220 (bright/pastel dataset). Red dominates slightly. Low pixel-level variance confirms that colour augmentation (jitter, flip) would meaningfully increase training diversity."_


### Plot 6 — PCA → t-SNE Cluster Plot

Flatten each image to 12 288 features → PCA(50 dims) → t-SNE(2D) → scatter coloured by class.

**What to look for:** are any classes cleanly separated in flat pixel space? If t-SNE shows clear class clusters, MLP has a chance. If everything is mixed together, no amount of FC layers will help — this motivates CNN.  
Expected: Fire and Water may show some partial structure (distinctive colours). Normal will scatter everywhere (highest intra-class diversity).


In [ ]:
# ── t-SNE of flat pixel vectors ───────────────────────────────────────────────
# n_per_class=40 → 360 points total — fast enough even on CPU (~10s), meaningful enough to read
# EDA always on full dataset (df_full), never on the FAST_RUN subset
fig = eda_plots.plot_pca_tsne(
    TRAIN_DIR, df_full,
    n_per_class=6 if FAST_RUN else 40,
    out_path=TASK_OUT_DIR / "plots" / "plot_pca_tsne.png",
)
plt.show()
plt.close(fig)


> **Finding — Pixel-space Separability:**  
> _Fill in after running. Note whether any class shows distinct clusters or whether all classes overlap. Example: "No clean clusters — all 9 classes heavily overlap in t-SNE space. Fire shows weak partial structure (orange pixels cluster slightly). This confirms MLP cannot linearly separate these classes in pixel space → motivates CNN spatial features."_


### EDA 7 — Dataset Normalisation Constants (from train split only)

Compute the actual pixel mean and std from the **training split rows only**.  
This is the correct way: stats computed on the split you train on, then applied identically to val/test. Computing stats on the full dataset (including val rows) would be a mild form of data leakage.

We also compare against ImageNet constants to confirm how close they are.


In [ ]:
# ── Normalisation stats from TRAIN split only ─────────────────────────────────
# We need the split indices first — reuse the same deterministic stratified split
# that get_train_val_loaders uses, so stats are from exactly the right rows.
from sklearn.model_selection import train_test_split as _tts
from src.config import SEED as _SEED, CLASSES as _CLASSES

_label_to_idx = {c: i for i, c in enumerate(_CLASSES)}
_all_labels   = [_label_to_idx[lbl] for lbl in df["label"]]
_train_idx, _ = _tts(list(range(len(df))), test_size=0.2, random_state=_SEED, stratify=_all_labels)
df_train_split = df.iloc[_train_idx].reset_index(drop=True)

train_mean, train_std = eda_plots.compute_dataset_stats(TRAIN_DIR, df_train_split)

print("Train-split pixel stats (normalised 0-1):")
for i, ch in enumerate(["R", "G", "B"]):
    print(f"  {ch}: mean={train_mean[i]:.4f}, std={train_std[i]:.4f}")
print(f"\nImageNet reference: mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]")
print(f"\nDifference vs ImageNet:")
imagenet_mean = [0.485, 0.456, 0.406]
for i, ch in enumerate(["R", "G", "B"]):
    diff = train_mean[i] - imagenet_mean[i]
    print(f"  {ch}: {diff:+.4f} ({'brighter' if diff > 0 else 'darker'} than ImageNet)")
print(f"\nConclusion: {'Using ImageNet constants is acceptable (< 0.10 difference).' if max(abs(train_mean[i] - imagenet_mean[i]) for i in range(3)) < 0.10 else 'Consider using dataset-specific constants — difference is significant.'}")


> **Finding — Normalisation:**  
> _Fill in after running. Record the 6 printed values and note the conclusion. Example: "Train-split mean=[0.62, 0.58, 0.54] — ~14% brighter than ImageNet. Difference is < 0.10 per channel, so using ImageNet constants is acceptable. We use them as-is in all transforms."_


---
## Part 2 — Model Experiments

### Setup
Two helper cells (below) define the shared infrastructure used by every experiment:
- **`build_loaders(augment, use_sampler)`** — wraps `get_train_val_loaders`, always uses `df` (the possibly-subsampled FAST_RUN set)
- **`run_experiment(model, name, criterion, optimizer, scheduler, loaders)`** — full train loop, early stopping, checkpoint save, evaluate best, store in `results_tracker`

Each experiment cell is then just 4 lines: instantiate model, criterion, optimizer, scheduler → call `run_experiment`.

### Experiment plan

| ID | Name | Model | Criterion | Optimizer | Expected effect |
|---|---|---|---|---|---|
| A | vanilla | 2-layer FC (128→64), no BN, no Dropout | CrossEntropy | Adam lr=1e-3 | reference point — should be worse |
| B | mlp_baseline | 3-layer (512→256→128), BN, Dropout(0.4) | CrossEntropy + class weights | Adam lr=1e-3 | our first run result (~0.193) |
| C | mlp_ls_drop03 | same as B, Dropout(0.3) | CrossEntropy + class weights + label_smoothing=0.1 | Adam lr=1e-3 | best expected regularisation combo |
| D | mlp_wd | same as C | same | Adam lr=1e-3, weight_decay=1e-4 | L2 reg reduces overfit |
| E | mlp_sampler | same as B | CrossEntropy + label_smoothing=0.1 | Adam | WeightedRandomSampler — may help minority classes |
| F | mlp_narrow | narrow+deep: 4 layers (256→128→64→32), BN, Dropout(0.3) | CrossEntropy + class weights + label_smoothing=0.1 | Adam lr=1e-3 | fewer params → less overfit? |
| G | mlp_bottleneck | bottleneck: expand-compress (512→1024→256→128), BN, Dropout(0.3) | CrossEntropy + class weights + label_smoothing=0.1 | Adam lr=1e-3 | wider middle captures more combinations? |

**Experiment order rationale:** regularisation changes (C–E) before architecture changes (F–G).  
We need to understand the overfitting baseline (B) and how much regularisation alone can recover it, before asking whether a different architecture shape helps.

### Architecture justification (MLP baseline)

| Design choice | Decision | Rationale |
|---|---|---|
| Input | Flatten 64×64×3 → 12 288 | MLP required — no convolutions |
| Hidden layers | 3 × (FC → BN → ReLU → Dropout) | depth adds capacity; 3 is enough |
| Width schedule | 512 → 256 → 128 | funnel forces compression |
| Batch Norm | after every FC | stabilises gradients with large flat input |
| Dropout | p=0.4 (default) | regularisation for ~6 M params on ~2880 train images |
| Output | FC(128 → 9), no softmax | `CrossEntropyLoss` applies log-softmax internally |
| Loss | CrossEntropyLoss(weight=class_weights) | inverse-frequency weights correct 2.76× imbalance |
| Optimiser | Adam lr=1e-3 | adaptive LR; robust default |
| Scheduler | StepLR(step_size=5, γ=0.5) | halves LR every 5 epochs |
| Early stopping | patience=5 on val_loss | saves best checkpoint, avoids overfitting |

### What NOT to try (and why)

- **More hidden layers / wider layers** — the problem is already severe overfitting (train_f1=0.55 vs val_f1=0.19). Adding parameters makes the gap worse, not better. We need more regularisation, not more capacity.
- **EPOCHS=50** — val_f1 plateaued at epoch 8 in the first run. Early stopping fired at epoch 17 (patience=5). More epochs waste GPU time and don't recover the val curve.
- **Grid search** — a 5×5 grid over {dropout, weight_decay, label_smoothing, LR, architecture} = 25 runs × 80s ≈ 33 min on Colab. Not worth it. The problem is overfitting — the relevant axis is **regularisation strength**. 4–5 manual targeted experiments guided by the previous result will teach us more than a blind grid sweep.
- **Skip connections in MLP** — skip connections (ResNet-style) help with gradient flow in very deep networks. Here we have at most 5 layers — gradient flow is not the bottleneck. The bottleneck is lack of spatial structure, which no amount of skip connections can fix.


In [ ]:
# ── Shared imports for all experiment cells ───────────────────────────────────
import torch.nn as nn
from torch.utils.data import DataLoader
from src.datasets.dataset import (
    PokemonDataset, compute_class_weights,
    get_base_transforms, get_train_val_loaders,
)
from src.models.mlp import MLP
from src.training.train import train_one_epoch, evaluate
from src.training.early_stopping import EarlyStopping
from src.evaluation.metrics import classification_report_str
from src.evaluation.plots import plot_history, plot_confusion_matrix

# build class weights from the run's df (correct proportions even for FAST_RUN)
label_to_idx     = {cls: i for i, cls in enumerate(CLASSES)}
_all_train_labels = [label_to_idx[lbl] for lbl in df["label"]]
class_weights    = compute_class_weights(_all_train_labels).to(device)

# test loader is always the full test set — not touched until submission
test_ds     = PokemonDataset(TEST_DIR, get_base_transforms(IMG_SIZE), csv_path=None)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print(f"Test images  : {len(test_ds)}")
print(f"Class weights: { {cls: round(class_weights[i].item(), 3) for i, cls in enumerate(CLASSES)} }")


# ── Helper: build_loaders ─────────────────────────────────────────────────────
# Call once per augment/sampler combination — reuse across experiments that share the same setup
def build_loaders(augment: bool = False, use_sampler: bool = False):
    """Return (train_loader, val_loader) for the current run's df and hyperparams."""
    return get_train_val_loaders(
        CSV_PATH, TRAIN_DIR, IMG_SIZE, BATCH_SIZE,
        augment=augment, use_sampler=use_sampler,
        num_workers=NUM_WORKERS, df_override=df,
    )


# ── Results tracker ───────────────────────────────────────────────────────────
# key = experiment name, value = metrics dict — filled by run_experiment
results_tracker: dict = {}


def _print_leaderboard(tracker: dict) -> None:
    """Print all experiments sorted by val_macro_f1 descending."""
    if not tracker:
        return
    rows = sorted(tracker.items(), key=lambda x: x[1]["val_macro_f1"], reverse=True)
    print(f"\n{'Rank':<5} {'Name':<22} {'val_F1':>8} {'val_acc':>8} {'epochs':>7} {'time(s)':>8}")
    print("-" * 60)
    for rank, (name, m) in enumerate(rows, 1):
        print(f"{rank:<5} {name:<22} {m['val_macro_f1']:>8.4f} {m['val_acc']:>8.4f} {m['total_epochs']:>7} {m['train_time_s']:>8.1f}")


# ── Helper: run_experiment ────────────────────────────────────────────────────
# One call per experiment — handles training, checkpointing, evaluation, and result logging.
def run_experiment(model, name: str, criterion, optimizer, scheduler, loaders) -> tuple:
    """
    Train model for up to EPOCHS epochs with early stopping.
    Saves best checkpoint to task1/outputs/checkpoints/<name>.pth.
    Stores metrics in results_tracker[name] and prints the leaderboard.
    Returns (model_with_best_weights, history_dict).
    """
    train_loader, val_loader = loaders
    ckpt_path = str(TASK_OUT_DIR / "checkpoints" / f"{name}.pth")
    stopper   = EarlyStopping(patience=PATIENCE, checkpoint_path=ckpt_path)
    history   = {"train_loss": [], "val_loss": [], "train_f1": [], "val_f1": []}
    t0        = time.time()

    print(f"\n{'='*60}")
    print(f"  Experiment: {name}")
    print(f"  Model params: {sum(p.numel() for p in model.parameters()):,}")
    print(f"{'='*60}")

    for epoch in range(1, EPOCHS + 1):
        ep_t0         = time.time()
        train_loss    = train_one_epoch(model, train_loader, criterion, optimizer, device)
        train_metrics = evaluate(model, train_loader, criterion, device)
        val_metrics   = evaluate(model, val_loader,   criterion, device)
        elapsed       = time.time() - ep_t0
        scheduler.step()

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_metrics["loss"])
        history["train_f1"].append(train_metrics["macro_f1"])
        history["val_f1"].append(val_metrics["macro_f1"])

        print(
            f"  Epoch {epoch:02d}/{EPOCHS} | "
            f"train_loss={train_loss:.4f}  train_f1={train_metrics['macro_f1']:.4f} | "
            f"val_loss={val_metrics['loss']:.4f}  val_f1={val_metrics['macro_f1']:.4f} | "
            f"{elapsed:.1f}s"
        )

        stopper(val_metrics["loss"], model)
        if stopper.stop:
            print(f"  Early stopping at epoch {epoch} (patience={PATIENCE}).")
            break

    total_time = time.time() - t0

    # load best weights and do a final clean evaluation
    model.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=True))
    best = evaluate(model, val_loader, criterion, device)

    results_tracker[name] = {
        "val_macro_f1": round(best["macro_f1"], 4),
        "val_acc":      round(best["acc"], 4),
        "val_loss":     round(best["loss"], 4),
        "total_epochs": len(history["train_loss"]),
        "train_time_s": round(total_time, 1),
        "history":      history,
    }

    print(f"\n  Best checkpoint: val_loss={best['loss']:.4f}  val_macro_f1={best['macro_f1']:.4f}")
    print(f"  Total time: {total_time:.1f}s ({total_time/len(history['train_loss']):.1f}s/epoch)")
    _print_leaderboard(results_tracker)

    return model, history


In [ ]:
# ── Experiment A — Vanilla Baseline ──────────────────────────────────────────
# Tiny 2-layer MLP, no BN, no Dropout, no class weights — a true baseline.
# Purpose: show that our main MLP design is already meaningfully better.

class VanillaMLP(nn.Module):
    """Flatten -> FC(12288->128) -> ReLU -> FC(128->64) -> ReLU -> FC(64->9). No BN, no Dropout."""
    def __init__(self, img_size: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(img_size * img_size * 3, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, NUM_CLASSES),
        )
    def forward(self, x):
        return self.net(x.view(x.size(0), -1))

loaders_base = build_loaders(augment=False, use_sampler=False)

model_A     = VanillaMLP(img_size=IMG_SIZE).to(device)
criterion_A = nn.CrossEntropyLoss()   # no class weights — true bare baseline
optimizer_A = torch.optim.Adam(model_A.parameters(), lr=LR)
scheduler_A = torch.optim.lr_scheduler.StepLR(optimizer_A, step_size=5, gamma=0.5)

model_A, history_A = run_experiment(model_A, "A_vanilla", criterion_A, optimizer_A, scheduler_A, loaders_base)


In [ ]:
# ── Experiment B — MLP Baseline (our main design) ─────────────────────────────
# 3-layer FC stack, BN, Dropout(0.4), weighted CrossEntropy.
# This mirrors the first Colab run. Run it again to get fresh metrics alongside A.

model_B     = MLP(img_size=IMG_SIZE, dropout=0.4).to(device)
criterion_B = nn.CrossEntropyLoss(weight=class_weights)
optimizer_B = torch.optim.Adam(model_B.parameters(), lr=LR)
scheduler_B = torch.optim.lr_scheduler.StepLR(optimizer_B, step_size=5, gamma=0.5)

model_B, history_B = run_experiment(model_B, "B_mlp_base", criterion_B, optimizer_B, scheduler_B, loaders_base)


In [ ]:
# ── Experiment C — Dropout(0.3) + label_smoothing=0.1 ─────────────────────────
# Key change: softer targets (label_smoothing) + slightly less regularisation (dropout 0.4→0.3).
# label_smoothing=0.1 distributes 0.1 of the probability mass to wrong classes,
# preventing the model from being overconfident — known to help imbalanced multi-class problems.

model_C     = MLP(img_size=IMG_SIZE, dropout=0.3).to(device)
criterion_C = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
optimizer_C = torch.optim.Adam(model_C.parameters(), lr=LR)
scheduler_C = torch.optim.lr_scheduler.StepLR(optimizer_C, step_size=5, gamma=0.5)

model_C, history_C = run_experiment(model_C, "C_ls01_drop03", criterion_C, optimizer_C, scheduler_C, loaders_base)


In [ ]:
# ── Experiment D — + weight_decay=1e-4 ───────────────────────────────────────
# Adds L2 regularisation to Experiment C. L2 penalises large weights,
# which should help reduce the train/val gap further.

model_D     = MLP(img_size=IMG_SIZE, dropout=0.3).to(device)
criterion_D = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
optimizer_D = torch.optim.Adam(model_D.parameters(), lr=LR, weight_decay=1e-4)
scheduler_D = torch.optim.lr_scheduler.StepLR(optimizer_D, step_size=5, gamma=0.5)

model_D, history_D = run_experiment(model_D, "D_wd1e4", criterion_D, optimizer_D, scheduler_D, loaders_base)


In [ ]:
# ── Experiment E — WeightedRandomSampler ─────────────────────────────────────
# Replaces class weights in loss with oversampling minority classes in the loader.
# Each epoch, Ground/Rock/Fighting images appear more frequently.
# Keep label_smoothing from Experiment C — that was independently helpful.

loaders_sampler = build_loaders(augment=False, use_sampler=True)

model_E     = MLP(img_size=IMG_SIZE, dropout=0.3).to(device)
criterion_E = nn.CrossEntropyLoss(label_smoothing=0.1)  # no class weights — sampler handles it
optimizer_E = torch.optim.Adam(model_E.parameters(), lr=LR, weight_decay=1e-4)
scheduler_E = torch.optim.lr_scheduler.StepLR(optimizer_E, step_size=5, gamma=0.5)

model_E, history_E = run_experiment(model_E, "E_sampler", criterion_E, optimizer_E, scheduler_E, loaders_sampler)


In [ ]:
# ── Experiment F — Narrow + Deep MLP ─────────────────────────────────────────
# Architecture: input -> 256 -> 128 -> 64 -> 32 -> 9  (4 hidden layers, max width 256)
# Hypothesis: fewer params per layer → less overfit on 2880 train images.
# Best regularisation combo from D: dropout=0.3 + label_smoothing + weight_decay.
# Run after E so we can compare architecture change vs regularisation change cleanly.
from src.models.mlp import NarrowMLP

model_F     = NarrowMLP(img_size=IMG_SIZE, dropout=0.3).to(device)
criterion_F = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
optimizer_F = torch.optim.Adam(model_F.parameters(), lr=LR, weight_decay=1e-4)
scheduler_F = torch.optim.lr_scheduler.StepLR(optimizer_F, step_size=5, gamma=0.5)

model_F, history_F = run_experiment(model_F, "F_narrow", criterion_F, optimizer_F, scheduler_F, loaders_base)


In [ ]:
# ── Experiment G — Bottleneck MLP ────────────────────────────────────────────
# Architecture: input -> 512 -> 1024 -> 256 -> 128 -> 9  (expand then compress)
# Hypothesis: the wide middle (1024 units) captures cross-pixel combinations before
# compressing down — the classic bottleneck pattern. Still no spatial structure.
# Same regularisation as D/F — isolating the architecture change.
from src.models.mlp import BottleneckMLP

model_G     = BottleneckMLP(img_size=IMG_SIZE, dropout=0.3).to(device)
criterion_G = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
optimizer_G = torch.optim.Adam(model_G.parameters(), lr=LR, weight_decay=1e-4)
scheduler_G = torch.optim.lr_scheduler.StepLR(optimizer_G, step_size=5, gamma=0.5)

model_G, history_G = run_experiment(model_G, "G_bottleneck", criterion_G, optimizer_G, scheduler_G, loaders_base)


---
## Part 3 — Comparison, Final Model & Evaluation


In [ ]:
# ── Results table + bar chart ─────────────────────────────────────────────────
import matplotlib.patches as mpatches

print("=== All Experiments — Sorted by Val Macro-F1 ===\n")
_print_leaderboard(results_tracker)

# bar chart of val_macro_f1 per experiment — easy to paste into slides
names  = list(results_tracker.keys())
f1s    = [results_tracker[n]["val_macro_f1"] for n in names]
times  = [results_tracker[n]["train_time_s"] for n in names]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# F1 bar chart
colors = ["steelblue" if f1 < max(f1s) else "darkorange" for f1 in f1s]
bars0 = axes[0].bar(names, f1s, color=colors)
axes[0].axhline(0.111, color="red", linestyle="--", linewidth=1, label="random baseline (0.111)")
axes[0].set_ylabel("Val Macro F1")
axes[0].set_title("Experiment Comparison — Val Macro F1")
axes[0].set_ylim(0, max(f1s) * 1.3)
axes[0].legend(fontsize=8)
axes[0].tick_params(axis="x", rotation=45)
# label sits just above each bar — offset is 2% of the y-axis range
y_range0 = max(f1s) * 1.3
for bar, v in zip(bars0, f1s):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + y_range0 * 0.02,
        f"{v:.4f}", ha="center", va="bottom", fontsize=8,
    )

# training time bar chart
bars1 = axes[1].bar(names, times, color="slategray")
axes[1].set_ylabel("Training time (s)")
axes[1].set_title("Training Time per Experiment")
axes[1].tick_params(axis="x", rotation=45)
# same trick: offset = 2% of y range so labels sit cleanly above bars
y_range1 = max(times) * 1.3 if max(times) > 0 else 1.0
axes[1].set_ylim(0, y_range1)
for bar, v in zip(bars1, times):
    axes[1].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + y_range1 * 0.02,
        f"{v:.1f}s", ha="center", va="bottom", fontsize=8,
    )

fig.tight_layout()
fig.savefig(TASK_OUT_DIR / "plots" / "task1_experiment_comparison.png", bbox_inches="tight", dpi=120)
plt.show()
plt.close(fig)


In [ ]:
# ── Load best experiment + full classification report ─────────────────────────
# Pick the experiment with the highest val_macro_f1 automatically
from src.models.mlp import MLP, NarrowMLP, BottleneckMLP   # ensure all classes are in scope

best_name = max(results_tracker, key=lambda k: results_tracker[k]["val_macro_f1"])
best_ckpt = str(TASK_OUT_DIR / "checkpoints" / f"{best_name}.pth")
print(f"Best experiment: {best_name}  (val_macro_f1={results_tracker[best_name]['val_macro_f1']:.4f})")

# rebuild the best model architecture — infer from experiment name
if best_name == "A_vanilla":
    best_model     = VanillaMLP(img_size=IMG_SIZE).to(device)
    best_criterion = nn.CrossEntropyLoss()
elif best_name == "E_sampler":
    best_model     = MLP(img_size=IMG_SIZE, dropout=0.3).to(device)
    best_criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
elif best_name in ("C_ls01_drop03", "D_wd1e4"):
    best_model     = MLP(img_size=IMG_SIZE, dropout=0.3).to(device)
    best_criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
elif best_name == "F_narrow":
    best_model     = NarrowMLP(img_size=IMG_SIZE, dropout=0.3).to(device)
    best_criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
elif best_name == "G_bottleneck":
    best_model     = BottleneckMLP(img_size=IMG_SIZE, dropout=0.3).to(device)
    best_criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
else:  # B_mlp_base
    best_model     = MLP(img_size=IMG_SIZE, dropout=0.4).to(device)
    best_criterion = nn.CrossEntropyLoss(weight=class_weights)

best_model.load_state_dict(torch.load(best_ckpt, map_location=device, weights_only=True))
val_loader_final = build_loaders()[1]  # fresh val loader for final eval

# collect all val predictions for report + confusion matrix
best_model.eval()
all_preds, all_labels_list = [], []
with torch.no_grad():
    for imgs, labels in val_loader_final:
        imgs = imgs.to(device)
        preds = best_model(imgs).argmax(dim=1).cpu().tolist()
        all_preds.extend(preds)
        all_labels_list.extend(labels.tolist())

val_metrics_final = evaluate(best_model, val_loader_final, best_criterion, device)
print(f"\nFinal val_loss={val_metrics_final['loss']:.4f}  val_acc={val_metrics_final['acc']:.4f}  macro_f1={val_metrics_final['macro_f1']:.4f}\n")
print(classification_report_str(all_labels_list, all_preds, CLASSES))


In [ ]:
# ── Training curves + confusion matrix for best experiment ────────────────────
best_history = results_tracker[best_name]["history"]

fig = plot_history(best_history, TASK_OUT_DIR / "plots" / "task1_history.png", title=best_name)
plt.show(); plt.close(fig)

fig = plot_confusion_matrix(all_labels_list, all_preds, CLASSES, TASK_OUT_DIR / "plots" / "task1_confusion.png")
plt.show(); plt.close(fig)


---
## Part 4 — Data Augmentation Sanity Check

**Hypothesis:** Data augmentation (`RandomHorizontalFlip` + `ColorJitter`) should NOT meaningfully help an MLP — and we prove it empirically here.

**Why it doesn't help MLP (theoretical):**  
A `RandomHorizontalFlip` swaps pixel position (0, 0) with position (0, W-1). After flattening, the model sees a completely different 12 288-dimensional vector. The MLP has no concept of spatial layout, so a flipped image provides no useful training signal — it looks like a different unrelated example.  
`ColorJitter` adds slight colour variation, which could marginally help by forcing slightly less overfit to exact pixel values, but the effect is small.

**Why we do this anyway:**
1. Empirically confirms the theory — makes the Task 1 → Task 2 story clean: "augmentation doesn't help MLP; it will help CNN."
2. Validates that the augmentation pipeline (`get_augment_transforms`) works end-to-end.
3. Prepares the loader infrastructure for Task 2 and Task 3 where augmentation is critical.

We take the **best experiment configuration** from Part 2 and retrain it with `augment=True`. Same model, same loss, same optimizer — only the training data loader changes.


In [ ]:
# ── Part 4 — Best model + augmentation ───────────────────────────────────────
# Retrain best experiment config with augment=True.
# Only the training loader changes — model, criterion, optimizer are identical.
# Expected: val_macro_f1 roughly equal or slightly worse than without augmentation.

loaders_aug = build_loaders(augment=True, use_sampler=False)

# rebuild a fresh best-model instance with the same architecture and criterion
if best_name == "A_vanilla":
    model_aug     = VanillaMLP(img_size=IMG_SIZE).to(device)
    criterion_aug = nn.CrossEntropyLoss()
elif best_name == "E_sampler":
    model_aug     = MLP(img_size=IMG_SIZE, dropout=0.3).to(device)
    criterion_aug = nn.CrossEntropyLoss(label_smoothing=0.1)
elif best_name in ("C_ls01_drop03", "D_wd1e4"):
    model_aug     = MLP(img_size=IMG_SIZE, dropout=0.3).to(device)
    criterion_aug = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
elif best_name == "F_narrow":
    model_aug     = NarrowMLP(img_size=IMG_SIZE, dropout=0.3).to(device)
    criterion_aug = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
elif best_name == "G_bottleneck":
    model_aug     = BottleneckMLP(img_size=IMG_SIZE, dropout=0.3).to(device)
    criterion_aug = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
else:  # B_mlp_base
    model_aug     = MLP(img_size=IMG_SIZE, dropout=0.4).to(device)
    criterion_aug = nn.CrossEntropyLoss(weight=class_weights)

optimizer_aug = torch.optim.Adam(model_aug.parameters(), lr=LR, weight_decay=1e-4)
scheduler_aug = torch.optim.lr_scheduler.StepLR(optimizer_aug, step_size=5, gamma=0.5)

model_aug, history_aug = run_experiment(
    model_aug, f"{best_name}_augmented",
    criterion_aug, optimizer_aug, scheduler_aug, loaders_aug,
)

# compare augmented vs non-augmented side by side
f1_no_aug = results_tracker[best_name]["val_macro_f1"]
f1_aug    = results_tracker[f"{best_name}_augmented"]["val_macro_f1"]
delta     = f1_aug - f1_no_aug

print(f"\n=== Augmentation effect on best model ({best_name}) ===")
print(f"  Without augmentation : val_macro_f1 = {f1_no_aug:.4f}")
print(f"  With augmentation    : val_macro_f1 = {f1_aug:.4f}")
print(f"  Delta                : {delta:+.4f}  ({'improved' if delta > 0.005 else 'no meaningful gain' if abs(delta) <= 0.005 else 'slightly hurt'})")
print(f"\nConclusion: augmentation {'helps' if delta > 0.005 else 'does not meaningfully help'} MLP on this dataset.")
print("This is expected: MLP flattens the image, so a flipped image is an unrelated vector.")
print("Augmentation will be valuable for CNN (Task 2) where spatial structure is preserved.")


In [ ]:
# ── Save all results to disk ──────────────────────────────────────────────────
# task1_results.json contains everything needed for the report — download it from Colab.
from sklearn.metrics import f1_score as _sk_f1

per_class_f1_arr = _sk_f1(all_labels_list, all_preds, average=None, zero_division=0)

# total time across all experiments
total_experiment_time = sum(v["train_time_s"] for v in results_tracker.values())

output = {
    "best_experiment":    best_name,
    "val_macro_f1":       round(val_metrics_final["macro_f1"], 4),
    "val_accuracy":       round(val_metrics_final["acc"], 4),
    "val_loss":           round(val_metrics_final["loss"], 4),
    "per_class_f1":       {cls: round(float(per_class_f1_arr[i]), 4) for i, cls in enumerate(CLASSES)},
    "all_experiments":    {
        name: {k: v for k, v in m.items() if k != "history"}
        for name, m in results_tracker.items()
    },
    "total_experiment_time_s": round(total_experiment_time, 1),
    "config": {
        "FAST_RUN": FAST_RUN, "EPOCHS": EPOCHS, "LR": LR,
        "BATCH_SIZE": BATCH_SIZE, "PATIENCE": PATIENCE, "IMG_SIZE": IMG_SIZE,
    },
    "best_history": {k: [round(v, 4) for v in vals] for k, vals in best_history.items()},
}

results_path = TASK_OUT_DIR / "results" / "task1_results.json"
with open(results_path, "w") as f:
    json.dump(output, f, indent=2)

print(f"Results saved -> {results_path}")
print(f"\nBest: {best_name}  val_macro_f1={output['val_macro_f1']}  val_acc={output['val_accuracy']}")
print(f"Total experiment time: {total_experiment_time:.0f}s across {len(results_tracker)} experiments")
print(json.dumps({"per_class_f1": output["per_class_f1"]}, indent=2))


In [ ]:
# ── Generate + validate submission CSV ───────────────────────────────────────
from src.evaluation.submission import generate_submission, validate_submission

SUB_PATH = TASK_OUT_DIR / "results" / "submission_task1.csv"
generate_submission(best_model, test_loader, CLASSES, SUB_PATH, device)
validate_submission(SUB_PATH, expected_rows=900)
print(f"Submission saved -> {SUB_PATH}")


In [ ]:
# ── Download outputs zip (Colab only) ────────────────────────────────────────
# Run this cell after all experiments to get plots, checkpoints, and results JSON.
if IN_COLAB:
    import shutil
    from google.colab import files
    zip_path = Path(ROOT) / "task1_outputs"
    shutil.make_archive(str(zip_path), "zip", root_dir=str(TASK_OUT_DIR.parent), base_dir=TASK_OUT_DIR.name)
    files.download(str(zip_path) + ".zip")
else:
    print(f"Outputs at: {TASK_OUT_DIR.resolve()}")


---
## Summary — Full Colab Run (FAST_RUN=False)

| Metric | Value |
|---|---|
| Best experiment | **A_vanilla** |
| Val macro-F1 (best) | **0.2104** |
| Val accuracy (best) | **25.14%** |
| Kaggle public score | _(fill in after submission)_ |
| Epochs run (best exp) | **11 / 30** — best checkpoint at epoch 6 |
| Total experiment time | **674.6 s (~11 min)** — all 8 experiments |
| Best per-class F1 | **Fire: 0.4157** |
| Worst per-class F1 | **Rock: 0.0000** — model never predicts this class correctly |
| Main confusion pair | Rock/Ground → predicted as other classes |
| Augmentation effect | **−0.0259 F1** (0.2104 → 0.1845) — confirmed: augmentation hurts MLP |
| F_narrow early stop | ❌ Never fired — ran all 30 epochs (worst result: 0.1343) |

**GPU/resource report:**
- GPU: T4 (Colab free tier, 15 GB VRAM)
- Total wall-clock: **674.6 s (~11 min)**
- Per-epoch time (A_vanilla): **~4.5 s/epoch**
- Early stopping fired: ✅ on all experiments **except F_narrow**
- Memory usage: < 100 MB peak (MLP is tiny)

**Per-class F1 — all classes:**

| Class | F1 |
|---|---|
| Fire | 0.4157 |
| Water | 0.3410 |
| Poison | 0.2651 |
| Normal | 0.2370 |
| Grass | 0.2264 |
| Bug | 0.1831 |
| Fighting | 0.1732 |
| Ground | 0.0519 |
| Rock | 0.0000 |

**Key finding:** A_vanilla (simplest model — no BN, no Dropout, no class weights) outperformed all 6 regularised variants.
Lesson: on small datasets (~2880 images), over-regularisation is as harmful as under-regularisation.

**Next steps:**
- Submit `task1/outputs/results/submission_task1.csv` to Kaggle → record test macro-F1
- Try Experiment H (VanillaMLP_v2: wider 256-unit first layer, still no regularisation)
- Switch early stopping to monitor `val_f1` instead of `val_loss` (epoch 9 had better val_f1=0.218)
- Soft ensemble: average A_vanilla + G_bottleneck predictions (no retraining needed)
